# 3. Message retrieval for manual analysis (Section 4.2)

The paper indexed the messages in Elasticsearch and searched for the top named entities; the
retrieved messages were then read by a Theology specialist. This notebook does the same search
with pandas (no server needed) and counts how many times each message was shared
(the 53 / 48 / 8 counts in the paper).

**The output files contain message text, including personal information about a private person.
They are written to `results/03_message_retrieval/`, which is git-ignored. Do not publish them.**

In [ ]:
from utils import *
pd.set_option("display.max_colwidth", 150)

In [ ]:
def share_counts(df):
    return (df.groupby("text_content_anonymous")
              .agg(shares=("id_message", "size"), groups=("id_group_anonymous", "nunique"))
              .sort_values("shares", ascending=False).reset_index())

out = results_folder("03_message_retrieval")
datasets = {p: load_dataset(p) for p in ["whatsapp", "telegram"]}
most_shared = {p: share_counts(df) for p, df in datasets.items()}
for p, table in most_shared.items():
    table.head(50).to_csv(out / f"{NAMES[p]}_most_shared.csv", index=False)

## Most shared messages

In [ ]:
most_shared["whatsapp"].head(5)

In [ ]:
most_shared["telegram"].head(5)

## Messages containing the top named entities
Run notebook 02 first.

In [ ]:
entities = pd.read_csv(RESULTS / "02_named_entities" / "top_entities.csv")
for p, df in datasets.items():
    frames = []
    for entity in entities.query("dataset == @NAMES[@p]")["entity"].dropna().unique():
        if len(str(entity)) < 3:
            continue
        hits = df[df["text_processed"].str.contains(str(entity).lower(), regex=False)]
        frames.append(share_counts(hits).head(20).assign(query=entity))
    pd.concat(frames).to_csv(out / f"{NAMES[p]}_retrieved_by_entity.csv", index=False)
print("saved in", out)

## Comparison with the paper

In [ ]:
compare_with_paper([
    ("FW.Br most shared message (shares)", 53, most_shared["whatsapp"]["shares"].iloc[0]),
    ("FT.Br most shared message (shares)", 48, most_shared["telegram"]["shares"].iloc[0]),
])

## Optional: Elasticsearch index
To explore the messages interactively as in the paper, start Elasticsearch locally,
`pip install elasticsearch` and run the cell below.

In [ ]:
# from elasticsearch import Elasticsearch, helpers
# es = Elasticsearch("http://localhost:9200")
# for p, df in datasets.items():
#     helpers.bulk(es, ({"_index": NAMES[p].lower(), "_id": str(r.id_message),
#                        "_source": {"text": r.text_content_anonymous, "group": r.id_group_anonymous}}
#                       for r in df.itertuples()))